In [16]:
import os
import torch

gpu_list = [0]
gpu_list_str = ','.join(map(str, gpu_list))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", gpu_list_str)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [17]:
from torch.nn import Linear
from model.layers import Add, Clone
from model.ViT import Attention
import torch.nn.functional as F
from einops import rearrange
import torch.nn as nn
import torchvision.models as models
from torch_geometric.nn import GATv2Conv, LayerNorm
from model.ViT import Mlp, VisionTransformer


class GraphNN(nn.Module):
    def __init__(self, cell_dim=80):
        super(GraphNN, self).__init__()
        self.resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.resnet18 = torch.nn.Sequential(*list(self.resnet18.children())[:-1])
        
        self.embed_dim = 32 * 8
        self.head = 8
        self.dropout = 0.3
        
        self.conv1 = GATv2Conv(in_channels=512, out_channels=int(self.embed_dim/self.head), heads=self.head)
        self.norm1 = LayerNorm(in_channels=self.embed_dim)
        
        self.head = Mlp(in_features=self.embed_dim, hidden_features=512*2, out_features=cell_dim)
        
    def forward(self, x, edge_index):
        x_spot = self.resnet18(x)
        x_spot = x_spot.squeeze()
        
        x_local = self.conv1(x=x_spot, edge_index=edge_index)
        x_local = self.norm1(x_local)
        
        cell_prediction = self.head(x_local)
        
        return cell_prediction

In [18]:
from glob import glob
tif_list = glob('/data1/r20user3/shared_project/Hist2Cell/code/training/train_test_splits/humanlung_cell2location/test*')
tif_list.sort()
test_slides = list()
for tif in tif_list:
    tif_path = tif.split('_')[-1].split('.')[0]
    test_slides.append(tif_path)
test_slides

['A37', 'A42', 'A48', 'A50']

In [19]:
from torch_geometric.data import Batch
from torch_geometric.loader import NeighborLoader
import torch_geometric
torch_geometric.typing.WITH_PYG_LIB = False
from tqdm import tqdm
import numpy as np
from scipy.stats import pearsonr
import joblib
    
for case in test_slides:
    print(case)
    ensemble = True
    vit_depth = 3
    celltype_num = 80
    model = GraphNN(cell_dim=celltype_num)
    checkpoint = torch.load("/data1/r20user3/shared_project/Hist2Cell/code/training/model_weights/humanlung_epoch100_lr1e-4_2hop_rawimgGNN_onlycell_"+case+"_best_cell_all_abundance_average.pth")
    model.load_state_dict(checkpoint)
    model = model.to(device)
    
    test_set = "/data1/r20user3/shared_project/Hist2Cell/code/training/train_test_splits/humanlung_cell2location/test_leave_"+case+".txt"
    data_path = "/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/human_lung_cell2location"
    hop = 2
    subgraph_bs = 16   
    
    criterion = nn.MSELoss().to(device)
    
    test_slides = open(test_set).read().split('\n')

    test_graph_list = list()
    for item in test_slides:
        test_graph_list.append(torch.load(os.path.join(data_path, item+'.pt')))
    test_dataset = Batch.from_data_list(test_graph_list)
    
    print(test_slides)
    # continue    

    test_loader = NeighborLoader(
        test_dataset,
        num_neighbors=[-1]*hop,
        batch_size=subgraph_bs,
        directed=False,
        input_nodes=None,
        shuffle=False,
        # num_workers=8,
        # pin_memory=True, 
        # prefetch_factor=2,
    )

    with torch.no_grad():
        model.eval()

        test_sample_num = 0
        test_cell_pred_array = []
        test_cell_label_array = []
        test_cell_pos_array = []
        test_loss, test_cell_abundance_loss = 0, 0
        
        for graph in tqdm(test_loader):
            x = graph.x.to(device)
            y = graph.y.to(device)
            pos = graph.pos.to(device)
            edge_index = graph.edge_index.to(device)
            cell_label = y[:, 250:]
            cell_pred = model(x=x, edge_index=edge_index)
            cell_loss = criterion(cell_pred, cell_label)

            loss = cell_loss
                
            center_num = len(graph.input_id)
            center_cell_label = cell_label[:center_num, :]
            center_cell_pred = cell_pred[:center_num, :]
            center_cell_pos = pos[:center_num, :]
            
            test_cell_label_array.append(center_cell_label.squeeze().cpu().detach().numpy())
            test_cell_pred_array.append(center_cell_pred.squeeze().cpu().detach().numpy())
            test_cell_pos_array.append(center_cell_pos.squeeze().cpu().detach().numpy())
            test_sample_num = test_sample_num + center_num
            
            test_loss += loss.item() * center_num
            test_cell_abundance_loss += cell_loss.item() * center_num

        test_cell_abundance_loss = test_cell_abundance_loss / test_sample_num
    
        if len(test_cell_pred_array[-1].shape) == 1:
            test_cell_pred_array[-1] = np.expand_dims(test_cell_pred_array[-1], axis=0)
        test_cell_pred_array = np.concatenate(test_cell_pred_array)
        if len(test_cell_label_array[-1].shape) == 1:
            test_cell_label_array[-1] = np.expand_dims(test_cell_label_array[-1], axis=0)
        test_cell_label_array = np.concatenate(test_cell_label_array)
        if len(test_cell_pos_array[-1].shape) == 1:
            test_cell_pos_array[-1] = np.expand_dims(test_cell_pos_array[-1], axis=0)
        test_cell_pos_array = np.concatenate(test_cell_pos_array)
    
        test_cell_abundance_pos_pearson_average = 0.0
        test_cell_abundance_all_pearson_average = 0.0
        test_cell_abundance_pos_pearson_count = 0   
        test_cell_pearson_list = []
        for i in range(celltype_num):
            r, p = pearsonr(test_cell_pred_array[:, i], test_cell_label_array[:, i])
            if r > 0.0 and p <= 0.05:
                test_cell_abundance_pos_pearson_count = test_cell_abundance_pos_pearson_count + 1
                test_cell_abundance_pos_pearson_average = test_cell_abundance_pos_pearson_average + r
            if np.isnan(r):
                r = 0.0
            test_cell_abundance_all_pearson_average = test_cell_abundance_all_pearson_average + r
            test_cell_pearson_list.append(r)
        if test_cell_abundance_pos_pearson_count >= 1:
            test_cell_abundance_pos_pearson_average /= test_cell_abundance_pos_pearson_count
        else:
            test_cell_abundance_pos_pearson_average = 0.0
        test_cell_abundance_all_pearson_average = test_cell_abundance_all_pearson_average / celltype_num

        print(case)
        print("saving " + "best cell all abundance average " + str(test_cell_abundance_all_pearson_average))
        
    
    Predictions = dict()

    for slide_no in range(len(test_slides)):
        indices = np.where(test_dataset.batch.numpy() == slide_no)
        test_cell_pred_array_sub = test_cell_pred_array[indices]
        test_cell_label_array_sub = test_cell_label_array[indices]
        test_cell_pos_arraay_sub = test_cell_pos_array[indices]
        
        test_cell_abundance_pos_pearson_average = 0.0
        test_cell_abundance_all_pearson_average = 0.0
        test_cell_abundance_pos_pearson_count = 0   
        test_cell_pearson_list = []
        for i in range(celltype_num):
            r, p = pearsonr(test_cell_pred_array_sub[:, i], test_cell_label_array_sub[:, i])
            if r > 0.0 and p <= 0.05:
                test_cell_abundance_pos_pearson_count = test_cell_abundance_pos_pearson_count + 1
                test_cell_abundance_pos_pearson_average = test_cell_abundance_pos_pearson_average + r
            if np.isnan(r):
                r = 0.0
            test_cell_abundance_all_pearson_average = test_cell_abundance_all_pearson_average + r
            test_cell_pearson_list.append(r)
        if test_cell_abundance_pos_pearson_count >= 1:
            test_cell_abundance_pos_pearson_average /= test_cell_abundance_pos_pearson_count
        else:
            test_cell_abundance_pos_pearson_average = 0.0
        test_cell_abundance_all_pearson_average = test_cell_abundance_all_pearson_average / celltype_num
        
        Predictions[test_slides[slide_no]] = {
            'cell_abundance_predictions': test_cell_pred_array_sub,
            'cell_abundance_labels': test_cell_label_array_sub,
            'coords': test_cell_pos_arraay_sub,
        }
        
        print(test_slides[slide_no]) 
        print(test_cell_abundance_all_pearson_average)    

    save_path = "/data1/r20user3/shared_project/Hist2Cell/code/analysis/inference/humanlung/humanlung_epoch100_lr1e-4_2hop_rawimgGNN_onlycell_"+case+"_best_cell_all_abundance_average.pkl"
    joblib.dump(Predictions, save_path)

A37
['WSA_LngSP9258464', 'WSA_LngSP9258468', 'WSA_LngSP10193347', 'WSA_LngSP10193348']


100%|██████████| 468/468 [00:51<00:00,  9.12it/s]


A37
saving best cell all abundance average 0.41153401802814144
WSA_LngSP9258464
0.2999081156300151
WSA_LngSP9258468
0.2756004005945944
WSA_LngSP10193347
0.22970422642687688
WSA_LngSP10193348
0.2991673720550883
A42
['WSA_LngSP8759311', 'WSA_LngSP8759312', 'WSA_LngSP8759313']


100%|██████████| 426/426 [00:47<00:00,  8.89it/s]


A42
saving best cell all abundance average 0.30671015642617594
WSA_LngSP8759311
0.3940745356588845
WSA_LngSP8759312
0.36970542350345204
WSA_LngSP8759313
0.1889974298316574
A48
['WSA_LngSP10193345', 'WSA_LngSP10193346']


100%|██████████| 354/354 [00:42<00:00,  8.39it/s]


A48
saving best cell all abundance average 0.25059150928951757
WSA_LngSP10193345
0.21611019069612966
WSA_LngSP10193346
0.20792068238992897
A50
['WSA_LngSP9258463', 'WSA_LngSP9258467']


100%|██████████| 51/51 [00:04<00:00, 11.76it/s]


A50
saving best cell all abundance average 0.33737845284933043
WSA_LngSP9258463
0.29287719809868845
WSA_LngSP9258467
0.39819264599774945


In [20]:
test_slides

['WSA_LngSP9258463', 'WSA_LngSP9258467']